# Nearest Neighbour Classification

In [1]:
import scipy.io
import numpy as np
import sklearn
from sklearn.feature_extraction import image
import matplotlib.pyplot as plt
from helper_functions import *

## Dataset - Data

In [2]:
data = scipy.io.loadmat('Data/data.mat')
faces = np.array(data['face'])
#number of classes 
M = 200
N = 600 # 3 per class

   ### Split Data

In [3]:

#select images 2 and 3 from each subject for training
faces_flat = faces.reshape(-1,faces.shape[2])
#print(faces_flat.shape)
train_idx =[]
test_idx = []
for n in range(M):
    idx = [3*n+1, 3*n+2]
    train_idx = train_idx + idx
    test_idx = test_idx + [3*n]
#print(test_idx)
training_faces = faces_flat[:,train_idx]
test_faces = faces_flat[:,test_idx]
#print(training_faces.shape)
#print(test_faces.shape)



### Perform PCA

In [4]:
#reduce dimensions from 504 to 50 (only 400 obsercations, sigma_w will be non invertible for 504).
A, train_mean = PCA(training_faces, 100)
X_PCA = A @ (training_faces - train_mean)
X_test_PCA = A @ (test_faces - train_mean)
print(training_faces.shape)


(504, 400)


### Perform MDA

In [5]:
#reduce dimensions from 100 to 50
A,class_means,sigma_w = MDA(X_PCA, 40, M)
X_MDA = A @ X_PCA
X_test_MDA = A @ X_test_PCA
print(X_MDA.shape)
################# Label the data sets
#training data
y_train = np.array([[i for i in range(200) for _ in range(2)]])
X_MDA = np.vstack((y_train,X_MDA))
#add this as first row of training dataset
y_test = np.array([[i for i in range(200)]])
X_test_MDA = np.vstack((y_test,X_test_MDA))

(40, 400)


### k - NN Classifier

In [21]:
for k in range(1,10):
    ### training error
    nearest, neighbours = k_NN(X_MDA, X_MDA, k)
    error = 1/y_train.shape[1]*sum(a!=b for (a,b) in zip(nearest.flatten(),y_train.flatten()))
    print(f"Training error for k = {k} is {error}")
   
    ### test error
    nearest, neighbours = k_NN(X_MDA, X_test_MDA, k)
    error = 1/y_test.shape[1]*sum(a!=b for (a,b) in zip(nearest.flatten(),y_test.flatten()))
    print(f"Testing error for k = {k} is {error}\n")
### training error
k = 399
nearest, neighbours = k_NN(X_MDA, X_MDA, k)
error = 1/y_train.shape[1]*sum(a!=b for (a,b) in zip(nearest.flatten(),y_train.flatten()))
print(f"Training error for k = {k} is {error}")

### test error
nearest, neighbours = k_NN(X_MDA, X_test_MDA, k)
error = 1/y_test.shape[1]*sum(a!=b for (a,b) in zip(nearest.flatten(),y_test.flatten()))
print(f"Testing error for k = {k} is {error}\n")

Training error for k = 1 is 0.0
Testing error for k = 1 is 0.21

Training error for k = 2 is 0.0
Testing error for k = 2 is 0.23500000000000001

Training error for k = 3 is 0.0
Testing error for k = 3 is 0.245

Training error for k = 4 is 0.1
Testing error for k = 4 is 0.265

Training error for k = 5 is 0.225
Testing error for k = 5 is 0.32

Training error for k = 6 is 0.3
Testing error for k = 6 is 0.37

Training error for k = 7 is 0.365
Testing error for k = 7 is 0.425

Training error for k = 8 is 0.42
Testing error for k = 8 is 0.49

Training error for k = 9 is 0.49
Testing error for k = 9 is 0.515

Training error for k = 399 is 0.995
Testing error for k = 399 is 0.995



## Dataset - Pose

In [8]:
data = scipy.io.loadmat('Data/pose.mat')
faces = np.array(data['pose'])
M = 68

In [9]:
print(faces[:,:,0,0].shape)

(48, 40)


### Split data

In [10]:
faces_flat = faces.reshape(-1, faces.shape[2], faces.shape[3])  # flatten images

train_idx = list(range(0,9))
test_idx = list(range(9,13))

# Vectorized selection
# training_faces: (pixels, 9 images * 68 subjects) first 9 images of subj 0, and so on
training_faces = faces_flat[:, train_idx, :].swapaxes(1, 2).reshape(faces_flat.shape[0], -1)

# test_faces: (pixels, 4 images * 68 subjects)
test_faces = faces_flat[:, test_idx, :].swapaxes(1, 2).reshape(faces_flat.shape[0], -1)

print("Training faces shape:", training_faces.shape)
print("Test faces shape:", test_faces.shape)

Training faces shape: (1920, 612)
Test faces shape: (1920, 272)


### PCA and MDA

In [11]:
#PCA - reduce dimensions from 504 to 100 (only 400 obsercations, sigma_w will be non invertible for 504).
A = PCA(training_faces, 70)
X_PCA = A @ training_faces
X_test_PCA = A @ test_faces
print(X_PCA.shape)

#MDA - reduce dimensions from 100 to 50
A,class_means,sigma_w = MDA(X_PCA, 40, M)
A = np.real(A)
X_MDA = A @ X_PCA
X_test_MDA = A @ X_test_PCA
print(X_MDA.shape)

(70, 612)
(40, 612)


### Estimation

In [12]:
#after MDA, assumtion is all classes share covariance A@sigma_w@A.T, with means A@class_means
print(class_means.shape)
class_means = A @ class_means
sigma_w = A @ sigma_w @ A.T
print(sigma_w.shape)

(70, 68)
(40, 40)


### Bayes Classifier

In [13]:
# For shared covariance and equal priors, the classifier is the minimum Mahalanobis distance classifier
# training error
N_train = 612
sigma_inv = np.linalg.inv(sigma_w)
#find mahalanobis distances for training data
distances = np.zeros((N_train,M))
for i in range(M):
    delta = X_MDA - class_means[:,i].reshape(-1,1)
    #vectorized
    sigma_delta = sigma_inv @ delta
    distances[:,i] = np.sum(delta * sigma_delta, axis = 0)
classified = np.argmin(distances, axis = 1)
y_values = [i for i in range(68) for _ in range(9)]

error = 1/len(y_values)*sum(a!=b for (a,b) in zip(classified,y_values))
print(error)

N_test = 272
#find mahalanobis distances for training data
distances = np.zeros((N_test,M))
for i in range(M):
    delta = X_test_MDA - class_means[:,i].reshape(-1,1)
    #vectorized
    sigma_delta = sigma_inv @ delta
    distances[:,i] = np.sum(delta * sigma_delta, axis = 0)
classified = np.argmin(distances, axis = 1)
y_values = [i for i in range(68) for _ in range(4)]

error = 1/len(y_values)*sum(a!=b for (a,b) in zip(classified,y_values))
print(error)


0.0032679738562091504
0.1764705882352941


## Dataset - Illumination

In [14]:
data = scipy.io.loadmat('Data/illumination.mat')
faces_flat = np.array(data['illum'])
#number of classes 
M = 68

### Split Data

In [15]:
train_idx = list(range(0,15))
test_idx = list(range(15,21))

# Vectorized selection
# training_faces: (pixels, 9 images * 68 subjects) first 9 images of subj 0, and so on
training_faces = faces_flat[:, train_idx, :].swapaxes(1, 2).reshape(faces_flat.shape[0], -1)

# test_faces: (pixels, 4 images * 68 subjects)
test_faces = faces_flat[:, test_idx, :].swapaxes(1, 2).reshape(faces_flat.shape[0], -1)

print("Training faces shape:", training_faces.shape)
print("Test faces shape:", test_faces.shape)

Training faces shape: (1920, 1020)
Test faces shape: (1920, 408)


### PCA and MDA

In [16]:
#PCA - reduce dimensions from 504 to 100 (only 400 obsercations, sigma_w will be non invertible for 504).
A = PCA(training_faces, 300)
X_PCA = A @ training_faces
X_test_PCA = A @ test_faces
print(X_PCA.shape)


#MDA - reduce dimensions from 100 to 50
A,class_means,sigma_w = MDA(X_PCA, 40, M)
A = np.real(A)
X_MDA = A @ X_PCA
X_test_MDA = A @ X_test_PCA
print(X_MDA.shape)

(300, 1020)
(40, 1020)


### Estimation

In [17]:
#after MDA, assumtion is all classes share covariance A@sigma_w@A.T, with means A@class_means
print(class_means.shape)
class_means = A @ class_means
sigma_w = A @ sigma_w @ A.T
print(sigma_w.shape)

(300, 68)
(40, 40)


### Bayes Classifier

In [18]:
# For shared covariance and equal priors, the classifier is the minimum Mahalanobis distance classifier
# training error
N_train = 1020
sigma_inv = np.linalg.inv(sigma_w)
#find mahalanobis distances for training data
distances = np.zeros((N_train,M))
for i in range(M):
    delta = X_MDA - class_means[:,i].reshape(-1,1)
    #vectorized
    sigma_delta = sigma_inv @ delta
    distances[:,i] = np.sum(delta * sigma_delta, axis = 0)
classified = np.argmin(distances, axis = 1)
y_values = [i for i in range(68) for _ in range(15)]

error = 1/len(y_values)*sum(a!=b for (a,b) in zip(classified,y_values))
print(error)

N_test = 408
#find mahalanobis distances for training data
distances = np.zeros((N_test,M))
for i in range(M):
    delta = X_test_MDA - class_means[:,i].reshape(-1,1)
    #vectorized
    sigma_delta = sigma_inv @ delta
    distances[:,i] = np.sum(delta * sigma_delta, axis = 0)
classified = np.argmin(distances, axis = 1)
y_values = [i for i in range(68) for _ in range(6)]

error = 1/len(y_values)*sum(a!=b for (a,b) in zip(classified,y_values))
print(error)


0.0
0.0


In [19]:
a = np.ones((10,10))
b = np.zeros((10,10))
c = np.sum(a,axis = 0)
print(c)

[10. 10. 10. 10. 10. 10. 10. 10. 10. 10.]


In [20]:
a[:,0] = np.sum(a,axis = 0).reshape(1,-1)

In [21]:
print(a)

[[10.  1.  1.  1.  1.  1.  1.  1.  1.  1.]
 [10.  1.  1.  1.  1.  1.  1.  1.  1.  1.]
 [10.  1.  1.  1.  1.  1.  1.  1.  1.  1.]
 [10.  1.  1.  1.  1.  1.  1.  1.  1.  1.]
 [10.  1.  1.  1.  1.  1.  1.  1.  1.  1.]
 [10.  1.  1.  1.  1.  1.  1.  1.  1.  1.]
 [10.  1.  1.  1.  1.  1.  1.  1.  1.  1.]
 [10.  1.  1.  1.  1.  1.  1.  1.  1.  1.]
 [10.  1.  1.  1.  1.  1.  1.  1.  1.  1.]
 [10.  1.  1.  1.  1.  1.  1.  1.  1.  1.]]


In [ ]:
print(a)